# Documentation

The sections of this chapter explained every function that they used. Every
project, however, soon needs a function that the book never mentions, or
uses a familiar one in an unfamiliar way. No book can record the details of
every function: the main module of a deep learning framework defines several
hundred to a thousand public names, and new releases add names and deprecate
others. How, then, do we find out reliably what a function does?

The question matters because a call can look harmless and still do
something other than what we meant. The call `ones(2, 3)` looks like a
request for a $2 \times 3$ matrix of ones, but whether it is one depends on
how the function reads its arguments, and only the function's signature says
what each argument means.

In JAX, `jnp.ones(2, 3)` raises an error, while `jnp.ones((2, 3))` returns
the matrix.

Instead of a list of functions, this section gives a procedure of four
steps, shown in the figure. We discover the names that a library offers,
inspect the arguments and the documentation of a candidate, read the
reference pages or the source code when the documentation leaves a question
open, and verify the behavior with a small example whose result we
predicted. All four steps work inside a notebook, and at the end of the
section they explain the result above.

![Four steps for finding out what a function does: discover the names that a module defines, inspect the signature and docstring of a candidate, read the reference documentation or the source code, and verify the behavior with a small example. A result that contradicts the prediction, or answers a question that the prediction left open, sends us back to read further.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/lookup-api-discovery-loop.svg)

We import the framework and the function `pprint`, which prints long lists
compactly.

In [1]:
from pprint import pprint
import jax
from jax import numpy as jnp

## Discovering Names

Suppose that we need random samples from a distribution other than the
normal distribution, and we want to know which distributions the library
can sample from. We know roughly where to look, in the module for random
numbers, but not what its functions are called. The built-in function `dir`
lists the names that a module defines. Names that begin with an underscore
belong to the internals of the module, so we drop them and print the first
twenty of the rest:

In [2]:
pprint([name for name in dir(jax.random)
        if not name.startswith('_')][:20], compact=True)

['PRNGKey', 'ball', 'bernoulli', 'beta', 'binomial', 'bits', 'categorical',
 'cauchy', 'chisquare', 'choice', 'clone', 'dirichlet', 'double_sided_maxwell',
 'exponential', 'f', 'fold_in', 'gamma', 'generalized_normal', 'geometric',
 'gumbel']


The names are samplers, such as `bernoulli`, `beta`, `cauchy`,
`exponential`, and `gamma`, along with `Key`, the type of a random key, and
`PRNGKey`, an older function that creates the key every JAX sampler takes. This book uses its
successor, `jax.random.key`, which comes later in the alphabet.

A notebook offers the same list while we type: writing the name of a module
followed by a dot and pressing Tab lists the names that the module defines,
narrowed to those that match the letters typed next. When we do not know
which module to look in, the search box of the framework's online reference
searches all of them.

## Reading Documentation

We set the samplers aside, since the first exercise completes the search
for a gamma sampler, and inspect a function that we think we already know:
`ones`, which we have used throughout this chapter. The built-in function
`help` prints the *docstring* of a function, the documentation written into
its code. A docstring says what the function does, what each argument means,
and what the function returns, and it often ends with examples:

In [3]:
help(jnp.ones)

Help on function ones in module jax.numpy:

ones(shape: Any, dtype: Union[str, type[Any], numpy.dtype, jax._src.typing.SupportsDType, NoneType] = None, *, device: jaxlib._jax.Device | jaxlib._jax.Sharding | None = None, out_sharding: jax.sharding.NamedSharding | jax.P | None = None) -> jax.Array
    Create an array full of ones.

    JAX implementation of :func:`numpy.ones`.

    Args:
      shape: int or sequence of ints specifying the shape of the created array.
      dtype: optional dtype for the created array; defaults to float32 or float64
        depending on the X64 configuration (see :ref:`default-dtypes`).
      device: (optional) :class:`~jax.Device` or :class:`~jax.sharding.Sharding`
        to which the created array will be committed. This argument exists for
        compatibility with the :ref:`python-array-api`.
      out_sharding: (optional) :class:`~jax.sharding.PartitionSpec` or :class:`~jax.NamedSharding`
        representing the sharding of the created array (see `e

The first lines of the output show the function's *signature*, the list of
arguments that it accepts. Three conventions of Python help to read it. An
argument written as `name=value` has a default value and may be omitted.
Arguments that follow a lone `*` can be passed only by keyword. An argument
written as `*name` collects any number of positional arguments.

With these conventions, the signature predicts what `ones(2, 3)` does
before we run it:

The signature `ones(shape, dtype=None, *, device=None, ...)` takes the
shape as a single argument and the data type as the second positional
argument. In `jnp.ones(2, 3)`, the 2 therefore becomes the shape and the 3
the data type. The integer 3 is not a data type, so we predict an error.

In a notebook, two shortcuts show the same documentation. Appending `?` to
a name, as in `ones?`, shows the docstring in a separate pane, and appending
`??` also shows the source code of the function. The source answers
questions that a docstring leaves open, such as how an argument is converted
or checked. Only functions written in Python have source code to show. Many
core functions are compiled from C++, and for those `??` shows the
docstring alone.


When the docstring is not enough, the framework's reference documentation
describes every public function on a page of its own, and its tutorials show
the functions in use.

For JAX, these are the
[API reference](https://docs.jax.dev/en/latest/jax.html) and the
[tutorials](https://docs.jax.dev/en/latest/tutorials.html).

## Verifying Behavior

Our reading of a docstring can be wrong, and occasionally the docstring
itself is out of date. A small example settles the question, provided that
we predict its result before we run it, so that the run could prove us
wrong. A run that merely completes without an error proves little. We
therefore test the prediction about `ones(2, 3)` by printing the shape and
the data type of the result, or the error if the call fails:

In [ ]:
try:
    x = jnp.ones(2, 3)
    print('shape:', x.shape, 'dtype:', x.dtype)
except TypeError as e:
    print('TypeError:', str(e).splitlines()[0])
print('ones((2, 3)) has shape', jnp.ones((2, 3)).shape)

The prediction holds: JAX cannot interpret the integer 3 as a data type, so
the call fails. We print only the first line of the message, which goes on
to name the likely intent. The
last line of the output shows the call that we meant: `jnp.ones((2, 3))`,
with the shape passed as one tuple, returns a $2 \times 3$ matrix.

More generally, when a result contradicts the prediction, or answers a
question that the prediction left open, we go around the loop again: we
read further, in the reference documentation or the source, and test the
new prediction.

Coding assistants answer many such questions directly. Asked for a
$2 \times 3$ tensor of ones, an assistant proposes a call, and the call
usually looks plausible. Such a suggestion is a candidate, and it goes
through the same steps as a name we found ourselves: we check that the name
exists, for example with `dir` or `hasattr`, read its signature, predict the
result, and run a small example. The last step matters most, because a
wrong suggestion need not fail: it can name a function that exists, fit
its signature, and run, yet compute something else.

The four steps let us find out what a function does without leaving the
notebook, and the last step makes the answer reliable, whether it came from
memory, from the documentation, or from an assistant. They complete the
tools of this chapter: tensors, tables, linear algebra, derivatives,
probability, and a way to find what the book leaves out.
that section uses these tools to build and train a first model.

## Summary

No text can record the details of every function in a deep learning framework,
and a call that looks harmless, such as `ones(2, 3)`, may read its arguments
differently than we assume; the signature says what each argument means. Four
steps settle such questions from within a notebook. `dir` and Tab completion
discover the names that a module defines, and `help` and `?` show the signature
and the docstring of a candidate. The reference documentation and `??` supply
the details that the docstring omits, and a small example whose result we
predicted verifies our reading. Suggestions from coding assistants are
candidates that go through the same steps.

## Exercises

1. [code] **Scale versus rate.** The gamma distribution with shape
   $k = 2$ and rate $\beta = 3$ has mean $k/\beta = 2/3$ and variance
   $k/\beta^2 = 2/9$. Use `dir` on `jax.random` to find its sampler and
   read the signature with `help` (or `?`) to find out how the sampler is
   parameterized. Draw 10,000 samples with $k = 2$ and $\beta = 3$, using
   an explicit key, and confirm the mean and variance from the samples.
1. [code] **Standard-deviation conventions.** For $n$ numbers whose
   squared deviations from their mean sum to $S$, a library computes the
   standard deviation either as $\sqrt{S/n}$ or as $\sqrt{S/(n-1)}$.
   that section standardized columns with the pandas method `std`.
   Compute the standard deviation of the values 1.0, 2.0, 3.0 and 4.0
   once with `pd.Series.std` and once with `jnp.std`. Use `help` on both
   functions to find out how each one sets the divisor, state the default
   divisor of each, and check both results against a hand computation.
   Then make the two results agree.
1. [code] **Checking a coding assistant's answer.** Ask a coding
   assistant the two questions below. Check each function it names in
   three steps: `dir` or `hasattr` shows that the name exists, `help` or
   `?` shows that the signature fits the suggested call, and a tiny
   example shows the claimed behavior.
    1. Ask how to split a JAX array into three equal parts along an axis.
    1. Ask for a single JAX function that reorders the axes of an array
       so that their lengths increase, an operation no library function
       provides. State which of the three checks, if any, each named
       function fails. Then build the operation from functions that pass
       all three checks and test it on an array of shape $(4, 2, 3)$.

4. [code] **Reading the source.** The plotting function `d2l.plot`,
   defined in that section, accepts its data in several forms:
   a single tensor, a tensor of $x$-values with one tensor of $y$-values,
   or a tensor of $x$-values with several curves. Import `d2l` as in that
   section, read the source of `d2l.plot` with `d2l.plot??` or your
   editor's go-to-definition, and find the lines that decide how many
   curves it draws and which values serve as their $x$-coordinates. Let
   `x` hold 50 evenly spaced values from $-2$ to $2$ (use `linspace`), and
   let `Y` be a matrix of shape $(3, 50)$.
   Predict from the source what `d2l.plot(x)`, `d2l.plot(x, x ** 2)`, and
   `d2l.plot(x, Y)` draw, in particular whether `Y` gives three curves of
   50 points or fifty curves of 3 points. Then run the three calls and
   compare.

1. [code] **Tests that can fail.** ● A verifying run is informative only if
   it would have given a different result had the claim been false.
    1. Find the framework's elementwise rounding function with `dir`, and
       design an input of at most four numbers on which three rules give
       three different results: round halves up, round halves away from
       zero, and round halves to the nearest even integer. Run the function
       on this input and state which rule the framework follows.
    1. A sort is *stable* if elements that compare equal keep their
       original order. Find the framework's `argsort`, and read what its
       docstring promises about stability. Then test stability on the input
       $(1, 0, 1, 0)$ and on 1,000 random integers from $\{0, 1\}$. Explain
       why a passing test on a short input cannot establish stability while
       a single failure refutes it, and state whether the behavior you
       observe agrees with the docstring.

[Discussions](https://d2l.discourse.group/t/17972)